In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark

# These are data_set

In [ ]:
orders = spark.createDataFrame(
    [
        (1001, 'c01', 'IN', 250.0, '2024-01-05'),
        (1002, 'c02', 'US', 40.0, '2024-01-06'),
        (1003, 'c01', 'IN', 120.0, '2024-01-07'),
        (1004, 'c03', 'UK', 500.0, '2024-01-07'),
        (1005, 'c02', 'US', 15.0, '2024-01-08'),
        (1006, 'c04', 'IN', 300.0, '2024-01-08'),
    ],
    ['order_id', 'customer_id', 'country', 'amount', 'order_date'],
)
orders.show()

+--------+-----------+-------+------+----------+
|order_id|customer_id|country|amount|order_date|
+--------+-----------+-------+------+----------+
|    1001|        c01|     IN| 250.0|2024-01-05|
|    1002|        c02|     US|  40.0|2024-01-06|
|    1003|        c01|     IN| 120.0|2024-01-07|
|    1004|        c03|     UK| 500.0|2024-01-07|
|    1005|        c02|     US|  15.0|2024-01-08|
|    1006|        c04|     IN| 300.0|2024-01-08|
+--------+-----------+-------+------+----------+



# Tasks

In [ ]:
# Q1 - Filter IN orders; columns order_id, customer_id, amount_inr; sort amount desc.

df = (
    orders
    .filter(F.col("country") == "IN")
    .withColumnRenamed("amount", "amount_inr")
    .select("order_id", "customer_id", "amount_inr")
    .orderBy(F.col("amount_inr").desc())
)

df.show()

+--------+-----------+----------+
|order_id|customer_id|amount_inr|
+--------+-----------+----------+
|    1006|        c04|     300.0|
|    1001|        c01|     250.0|
|    1003|        c01|     120.0|
+--------+-----------+----------+



# A2. Add order_size: Small <100, Medium <300, else Large.

In [ ]:
df = orders.withColumn(
    "order_size",
    F.when(F.col("amount") < 100, "Small")
     .when(F.col("amount") < 300, "Medium")
     .otherwise("Large")
)

df.show()

+--------+-----------+-------+------+----------+----------+
|order_id|customer_id|country|amount|order_date|order_size|
+--------+-----------+-------+------+----------+----------+
|    1001|        c01|     IN| 250.0|2024-01-05|    Medium|
|    1002|        c02|     US|  40.0|2024-01-06|     Small|
|    1003|        c01|     IN| 120.0|2024-01-07|    Medium|
|    1004|        c03|     UK| 500.0|2024-01-07|     Large|
|    1005|        c02|     US|  15.0|2024-01-08|     Small|
|    1006|        c04|     IN| 300.0|2024-01-08|     Large|
+--------+-----------+-------+------+----------+----------+



 # Comment which Operations are transformations vs actions; explore with show/count.

In [ ]:
# --------------------------------------------------------------------------
# TRANSFORMATIONS (Lazy - Builds execution logical plan, no data processed yet)
# --------------------------------------------------------------------------
df_filtered = orders.filter(F.col("country") == "IN")        # Transformation (Narrow)
df_renamed  = df_filtered.withColumnRenamed("amount", "amt")  # Transformation (Narrow)
df_sorted   = df_renamed.orderBy(F.col("amt").desc())         # Transformation (Wide/Shuffle)

# --------------------------------------------------------------------------
# ACTIONS (Triggers computation & executes the lineage plan)
# --------------------------------------------------------------------------
df_sorted.show()  # Action: Displays top 20 rows in driver output console
row_count = df_sorted.count()  # Action: Computes and returns integer count
print(f"Total rows in filtered set: {row_count}")

+--------+-----------+-------+-----+----------+
|order_id|customer_id|country|  amt|order_date|
+--------+-----------+-------+-----+----------+
|    1006|        c04|     IN|300.0|2024-01-08|
|    1001|        c01|     IN|250.0|2024-01-05|
|    1003|        c01|     IN|120.0|2024-01-07|
+--------+-----------+-------+-----+----------+

Total rows in filtered set: 3


# lazy evaluation; RDD vs DataFrame; why collect() is dangerous.
# Stretch. Function high_value_orders(df, min_amount)

In [ ]:
def high_value_orders(df, min_amount: float):
    """
    Filters a PySpark DataFrame for orders exceeding min_amount
    and returns the result ordered by amount descending.
    """
    return df.filter(F.col("amount") > min_amount).orderBy(F.col("amount").desc())

# Test the function with a threshold of 100.0
filtered_orders_df = high_value_orders(orders, min_amount=100.0)
filtered_orders_df.show()

+--------+-----------+-------+------+----------+
|order_id|customer_id|country|amount|order_date|
+--------+-----------+-------+------+----------+
|    1004|        c03|     UK| 500.0|2024-01-07|
|    1006|        c04|     IN| 300.0|2024-01-08|
|    1001|        c01|     IN| 250.0|2024-01-05|
|    1003|        c01|     IN| 120.0|2024-01-07|
+--------+-----------+-------+------+----------+

